# AIF360: Fairness in AI Toolkit

AIF360 es un conjunto de herramientas dise?adas para abordar la equidad y la justicia en los sistemas de inteligencia artificial (IA). Ofrece una serie de algoritmos, m?tricas y utilidades para ayudar a los desarrolladores y los profesionales de IA a comprender, medir y mitigar los sesgos y las disparidades en los modelos de IA.

## Utilizaci?n

- **Evaluar la equidad:** Proporciona m?tricas predefinidas y personalizables para evaluar la equidad de los modelos de IA en funci?n de atributos sensibles como g?nero, raza, edad, etc.
- **Mitigar sesgos:** Ofrece algoritmos y t?cnicas para mitigar los sesgos identificados en los modelos de IA, ya sea en el preprocesamiento de datos, durante el entrenamiento del modelo o en la etapa de postprocesamiento.
- **Comprender el impacto de las decisiones de IA:** Permite a los usuarios comprender c?mo los modelos de IA pueden afectar a diferentes grupos demogr?ficos y c?mo esos impactos pueden ser injustos o discriminatorios.
- **Promover la equidad y la justicia:** Facilita el dise?o de sistemas de IA m?s equitativos y justos al proporcionar herramientas para identificar y abordar los sesgos, lo que lleva a una toma de decisiones m?s inclusiva y ?tica.

#### Librerias requeridas

In [ ]:
from aif360.metrics import BinaryLabelDatasetMetric
from aif360.datasets import BinaryLabelDataset
from aif360.algorithms.preprocessing.reweighing import Reweighing
from sklearn.linear_model import LogisticRegression
from sklearn import preprocessing
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from IPython.display import Markdown, display
from aif360.algorithms.preprocessing.lfr import LFR
import pandas as pd

#### Lectura del dataset

In [ ]:
df = pd.read_csv('../data/Statlog_preprocesado.csv')

#### El AIF360 requiere de un dataset de tipo num?rico para poder trabajar, por lo que se realiza una conversi?n de las variables categ?ricas a num?ricas.

Se realiza un label encoder porque se ha comprobado que con one hot encoder el modelo pierde precisi?n en los pesos. Solamente se utilizar? este tipo de encoding en este notebook para poder trabajar con el AIF360, posteriormente se volver? a utilizar el dataset original con los pesos obtenidos con AIF360.

In [ ]:
categorical_columns = df.select_dtypes(include=['object', 'category']).columns

data_encoded = df.copy(deep=True)

#Use Scikit-learn label encoding to encode character data
lab_enc = preprocessing.LabelEncoder()
for col in categorical_columns:
        data_encoded[col] = lab_enc.fit_transform(df[col])
        le_name_mapping = dict(zip(lab_enc.classes_, lab_enc.transform(lab_enc.classes_)))
        print('Feature', col)
        print('mapping', le_name_mapping)

#### Se realiza un split del dataset en train y test

In [ ]:
X = data_encoded
X = X.drop('Class', axis=1)
y = data_encoded['Class']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

#### Se convierte el dataframe de Pandas en un objeto BinaryLabelDataset para poder trabajar con AIF360

Es necesario definir ciertos par?metros para poder realizar este tipo de conjunto de datos:

1. Asignaci?n de variable objetivo y sus valores

In [ ]:
target_label = ['Class']
favorable_labels = 1  # 'Concedido'
unfavorable_labels = 2   # 'Denegado'

2. Asignaci?n de variables protegidas y sus valores

Estas son variables que pueden generar alg?n sesgo en la sociedad como la edad, el genero, la raza...

In [ ]:
protected_attributes = [
    'Gender',
    'Foreign worker',
    'Marital Status',
    'Job'
]

privileged_groups = [
    {
        'Gender': 1, # Hombre
        'Foreign worker': 0, # 'no'
        'Marital Status': 2, # 'married'
        'Job': 0, # 'management/self-employed'
        'Job': 1, # 'skilled employee'
     }
]

unprivileged_groups = [
    {
        'Gender': 0,  # Mujer
        'Foreign worker': 1, # 'yes'
        'Marital Status': 0,  # 'divorced/separated'
        'Marital Status': 3,  # 'single'
        'Marital Status': 1,   # 'divorced/separated/married'
        'Job': 2,  # 'unemployed/non-resident'
        'Job': 3   # 'unskilled resident'
    }  
]


En este caso se eligen las siguientes variables como protegidas:
- Genero
- Trabajador extranjero (se entiende que es inmigrante)
- Estado civil
- Trabajo (Se entiende que tener _skills_ va directamente relacionado con el nivel de estudios)

In [ ]:
binary_dataset = BinaryLabelDataset(df=data_encoded,
                                      label_names=target_label,
                                      favorable_label=favorable_labels,
                                      unfavorable_label=unfavorable_labels,
                                      protected_attribute_names=protected_attributes
                                 )

#### M?tricas de equidad del conjunto de datos original

In [ ]:
# Metric for the original dataset
original_bias = BinaryLabelDatasetMetric(binary_dataset, 
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)
print("Diferencia de resultados medios entre grupos privilegiados y no privilegiados = %f" % original_bias.mean_difference())

#### Transformaci?n del conjunto de datos

Existen 4 tipos de transformaciones que se pueden realizar en el conjunto de datos:
- **Reweighing:** Modifica los pesos de las instancias para que el conjunto de datos sea m?s equitativo.
- **Disparate Impact Remover:** Elimina las caracter?sticas que pueden generar sesgo en el conjunto de datos.
- **LFR (Learning Fair Representations):** Aprende una representaci?n justa del conjunto de datos.
- **Data preprocessing:** Realiza un preprocesamiento de los datos para eliminar el sesgo.

Se ha comprobado que _Data preprocessing_ tarda demasiado como para poder realizarlo y _Dispare impact remover_ necesita el modelo entre medias y no es valido en este caso, por lo que se realizar? una transformaci?n con _Reweighing_ y _Disparate Impact Remover_.

In [ ]:
RW = Reweighing(unprivileged_groups=unprivileged_groups,
               privileged_groups=privileged_groups)
RW.fit(binary_dataset)
rw_binary_transformed = RW.transform(binary_dataset)

In [ ]:
TR = LFR(unprivileged_groups=unprivileged_groups,
         privileged_groups=privileged_groups,
         k=10, Ax=0.1, Ay=1.0, Az=2.0,
         verbose=1
        )
TR = TR.fit(binary_dataset, maxiter=5000, maxfun=5000)
tr_binary_transformed = TR.transform(binary_dataset)

#### M?tricas de equidad del conjunto de datos tras transformar los pesos

In [ ]:
rw_transformed_bias = BinaryLabelDatasetMetric(rw_binary_transformed, 
                                         unprivileged_groups=unprivileged_groups,
                                         privileged_groups=privileged_groups)
print("Diferencia de resultados medios entre grupos privilegiados y no privilegiados con RW = %f" % rw_transformed_bias.mean_difference())

In [ ]:
tr_transformed_bias = BinaryLabelDatasetMetric(tr_binary_transformed, 
                                         unprivileged_groups=unprivileged_groups,
                                         privileged_groups=privileged_groups)
print("Diferencia de resultados medios entre grupos privilegiados y no privilegiados con TR = %f" % tr_transformed_bias.mean_difference())

### Testing

#### Con el dataset con pesos REWEIGHING

In [ ]:
# Get the dataset and split into train and test
transformed_train, transformed_test = rw_binary_transformed.split([0.8], shuffle=True)

In [ ]:
# Logistic regression classifier and predictions
b_X_train = transformed_train.features
b_y_train = transformed_train.labels.ravel()
b_w_train = transformed_train.instance_weights.ravel()

b_X_test = transformed_test.features
b_y_test = transformed_test.labels.ravel()
b_w_test = transformed_test.instance_weights.ravel() 

model = LogisticRegression(max_iter=1000000)

model.fit(b_X_train, b_y_train, sample_weight=b_w_train)

b_y_pred = model.predict(b_X_test)

accuracy = accuracy_score(b_y_test, b_y_pred, sample_weight=b_w_test) 
print('Accuracy: %.2f' % (accuracy * 100))

#### Con el dataset con pesos LFR

In [ ]:
# Get the dataset and split into train and test
tr_X = tr_binary_transformed.convert_to_dataframe()[0]
tr_y = tr_X.get('Class')
tr_X = tr_X.drop('Class', axis=1)
b_X_train, b_X_test, b_y_train, b_y_test = train_test_split(tr_X, y, test_size=0.2, random_state=0)

In [ ]:
# Logistic regression classifier and predictions

model = LogisticRegression(max_iter=1000000)

model.fit(b_X_train, b_y_train)

b_y_pred = model.predict(b_X_test)

accuracy = accuracy_score(b_y_test, b_y_pred) 
print('Accuracy: %.2f' % (accuracy * 100))


#### Con el dataset sin cambios

In [ ]:
# read the original csv
df = pd.read_csv('../data/Statlog_preprocesado.csv')

categorical_column = df.select_dtypes(include=['object', 'category']).columns

data_encoded = df.copy(deep=True)
#Use Scikit-learn label encoding to encode character data
lab_enc = preprocessing.LabelEncoder()
for col in categorical_column:
        data_encoded[col] = lab_enc.fit_transform(df[col])
        le_name_mapping = dict(zip(lab_enc.classes_, lab_enc.transform(lab_enc.classes_)))

model = LogisticRegression(max_iter=1000000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print('Accuracy: %.2f' % (accuracy*100))

### Guardar los pesos

Se ha demostrado que el modelo con pesos REWEIGHING es el que mejor se ajusta a los datos, por lo que se guardan los pesos obtenidos para poder utilizarlos en el modelo final.

In [ ]:
# save to a csv binary_transformed.instance_weights
df['Weights'] = rw_binary_transformed.instance_weights
df.to_csv('../data/Statlog_weights.csv', index=False)